In [1]:
# Setup feedback system

import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor


def score_dataset(X, y, model=XGBRegressor()):
    # Label encoding for categoricals
    for colname in X.select_dtypes(["category", "object"]):
        X[colname], _ = X[colname].factorize()
    # Metric for Housing competition is RMSLE (Root Mean Squared Log Error)
    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="neg_mean_squared_log_error",
    )
    score = -1 * score.mean()
    score = np.sqrt(score)
    return score


# Prepare data
df = pd.read_csv("ames.csv")
X = df.copy()
y = X.pop("SalePrice")

In [2]:
# Das leere DataFrame wird erstellt
X_1 = pd.DataFrame()

# Verhältnis Wohnfläche zu Grundstück
X_1["LivLotRatio"] = X["GrLivArea"] / X["LotArea"]

# Geräumigkeit (Klammern beachten!)
X_1["Spaciousness"] = (X["FirstFlrSF"] + X["SecondFlrSF"]) / X["TotRmsAbvGrd"]

# Summe aller Außenflächen
X_1["TotalOutsideSF"] = (
    X["WoodDeckSF"]
    + X["OpenPorchSF"]
    + X["EnclosedPorch"]
    + X["Threeseasonporch"]
    + X["ScreenPorch"]
)

In [3]:
# YOUR CODE HERE
# One-hot encode BldgType. Use `prefix="Bldg"` in `get_dummies`
X_2 = pd.get_dummies(X.BldgType, prefix="Bldg")
# Multiply
X_2 = X_2.mul(X.GrLivArea, axis=0)

In [4]:
X_3 = pd.DataFrame()

# YOUR CODE HERE
X_3["PorchTypes"] = (
    X[
        [
            "WoodDeckSF",
            "OpenPorchSF",
            "EnclosedPorch",
            "Threeseasonporch",
            "ScreenPorch",
        ]
    ]
    .gt(0.0)
    .sum(axis=1)
)

In [5]:
df.MSSubClass.unique()

<StringArray>
[      'One_Story_1946_and_Newer_All_Styles',
                  'Two_Story_1946_and_Newer',
              'One_Story_PUD_1946_and_Newer',
      'One_and_Half_Story_Finished_All_Ages',
                               'Split_Foyer',
              'Two_Story_PUD_1946_and_Newer',
                       'Split_or_Multilevel',
                  'One_Story_1945_and_Older',
                'Duplex_All_Styles_and_Ages',
 'Two_Family_conversion_All_Styles_and_Ages',
    'One_and_Half_Story_Unfinished_All_Ages',
                  'Two_Story_1945_and_Older',
               'Two_and_Half_Story_All_Ages',
    'One_Story_with_Finished_Attic_All_Ages',
          'PUD_Multilevel_Split_Level_Foyer',
           'One_and_Half_Story_PUD_All_Ages']
Length: 16, dtype: str

In [6]:
X_4 = pd.DataFrame()

# YOUR CODE HERE
X_4["MSClass"] = X.MSSubClass.str.split("_", n=1, expand=True)[0]

In [7]:
X_5 = pd.DataFrame()

# YOUR CODE HERE
X_5["MedNhbdArea"] = X_5["MedNhbdArea"] = X.groupby("Neighborhood")[
    "GrLivArea"
].transform("median")

In [10]:
X_new = X.join([X_1, X_2, X_3, X_4, X_5])
print(score_dataset(X, y))
print(score_dataset(X_new, y))

0.14336778839535852


/var/folders/57/hy79q7hd0xvbx0p70vl8hw4m0000gn/T/ipykernel_82502/3023901455.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for colname in X.select_dtypes(["category", "object"]):


0.1395404037663999
